In [1]:
import os
import numpy as np
import pandas as pd
import cv2 as cv
from pathlib import Path
import warnings
from skimage.feature import hog
import tqdm
from sklearn.neighbors import KNeighborsClassifier
from sklearn import metrics
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import NearestNeighbors
warnings.filterwarnings("ignore")
pd.options.display.max_columns = None

In [ ]:
# all_images = []
# #labels = []
# def load_image(ids,path=image_folder):
#     img = cv.imread(image_folder+ids+'.jpg',cv.IMREAD_GRAYSCALE) #load at gray scale
#     #img = cv.cvtColor(img, cv.COLOR_BGR2GRAY) #convert to gray scale
#     return img,ids
# #20k samples were taken for modeling
# for ids in tqdm(list(styles.id)[:20000]):
#     img,ids = load_image(str(ids))
#     if img is not None:
#         all_images.append([img,int(ids)])
#     #labels.append(ids)
# len(all_images)


video_folder = "/media/osero/SamsungSSD/CMPE_SSD/frame-hand_left-c256_TOY"
labels = [] 
all_images = []
process_count = 0
for label_folder in os.listdir(video_folder):
    process_count += 1
    full_label_folder = os.path.join(video_folder, label_folder)
    label = int(label_folder)
    print("process_count: ", process_count, ' , label: ', label)
    for sample_folder in os.listdir(full_label_folder):
        full_sample_folder = os.path.join(full_label_folder, sample_folder)
        image_list = []
        for image_file in os.listdir(full_sample_folder):
            full_image_file = os.path.join(full_sample_folder, image_file)
            image = Image.open(full_image_file)
            image_list.append(image)
        all_images.append(image_list)
        labels.append(label)


abc = 4

In [ ]:
def resize_image(img,ids):
    return cv.resize(img, (60, 80),interpolation =cv.INTER_LINEAR)
    
all_images_resized = [[resize_image(x,y),y] for x,y in all_images]
len(all_images_resized)

In [ ]:
##HOG Descriptor
#Returns a 1D vector for an image
ppcr = 8
ppcc = 8
hog_images = []
hog_features = []
for image in tqdm(train_images):
    blur = cv.GaussianBlur(image,(5,5),0) #Gaussian Filtering
    fd,hog_image = hog(blur, orientations=8, pixels_per_cell=(ppcr,ppcc),cells_per_block=(2,2),block_norm='L2',visualize=True)
    hog_images.append(hog_image)
    hog_features.append(fd)
hog_features = np.array(hog_features)
hog_features.shape

In [ ]:
#normalization by 'L2-Hys'
from numpy import block


out = block / np.sqrt(np.sum(block ** 2) + eps ** 2)        
out = np.minimum(out, 0.2)        
out = out / np.sqrt(np.sum(out ** 2) + eps ** 2)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(hog_features,df_labels['class'],test_size=0.2,stratify=df_labels['class'])
print('Training data and target sizes: \n{}, {}'.format(X_train.shape,y_train.shape))
print('Test data and target sizes: \n{}, {}'.format(X_test.shape,y_test.shape))

In [ ]:
from sklearn.discriminant_analysis import StandardScaler


test_accuracy = []
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_train)
classifier = KNeighborsClassifier(n_neighbors=3,algorithm='brute')
classifier.fit(X_scaled, y_train)
test_accuracy = classifier.score(scaler.transform(X_test), y_test)
print(test_accuracy)